# Phase 8-bis — Que vaut vraiment ce 100 % ?

Le modèle GRPO a eu 100 % sur des problèmes **construits avec les mêmes 5 modèles de phrases** que l'entraînement. Ça prouve qu'il maîtrise *ces* problèmes. Ça ne prouve pas qu'il sait se servir d'une calculatrice.

On le soumet à quatre épreuves, en comparant à chaque fois le modèle **avant** et **après** GRPO :

| Épreuve | Question posée |
|---|---|
| A. Mêmes formulations, nouveaux nombres | Contrôle : on retrouve bien ~100 % ? |
| B. Formulations nouvelles | A-t-il appris la **tâche**, ou des **phrases** ? |
| C. Problèmes en deux étapes | Sait-il **enchaîner** les calculs ? |
| D. Calculs faciles (petits nombres) | Utilise-t-il l'outil **à bon escient**, ou partout ? |

**Kaggle :** GPU T4, Internet On.

⚠️ **Ordre important :** lance la cellule d'installation, puis **Run → Restart**, puis lance la suite **à partir de la cellule 2**. C'est le piège de versions de la dernière fois.

In [ ]:
!pip -q install -U transformers peft huggingface_hub
!pip -q uninstall -y torchao

In [1]:
import re, json, time, random
import numpy as np, torch
from getpass import getpass
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

login(token=getpass("Token Hugging Face (Write) : "))      # ton adaptateur GRPO est privé
device = "cuda"
MODELE, ADAPTATEUR = "Qwen/Qwen2.5-0.5B-Instruct", "Ezeck01/qwen2.5-0.5b-calculatrice-grpo"
tok = AutoTokenizer.from_pretrained(MODELE)
tok.padding_side = "left"
base = AutoModelForCausalLM.from_pretrained(MODELE, dtype=torch.float32).to(device)
modele = PeftModel.from_pretrained(base, ADAPTATEUR).to(device).eval()
print("GPU :", torch.cuda.get_device_name(0), "| modèle GRPO chargé")

Token Hugging Face (Write) :  ········


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

adapter_config.json:   0%|          | 0.00/1.23k [00:00<?, ?B/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 35.2MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

GPU : Tesla T4 | modèle GRPO chargé


/usr/local/lib/python3.13/dist-packages/peft/peft_model.py:642: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight', 'base_model.model.model.layers.0.mlp.gate_proj.lora_A.default.weight', 'base_model.model.model.layers.0.mlp.gate_proj.lora_B.default.weight', 'base_model.model.model.layers.0.mlp.up_proj.lora_A.default.weight', 'base_model.model.model.layers.0.mlp.up_proj.lora_B.default.we

In [7]:
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

poids = load_file(hf_hub_download(ADAPTATEUR, "adapter_model.safetensors"))
print("Clés du fichier :", list(poids)[:2])

lora = {n: p for n, p in modele.named_parameters() if "lora_" in n}
def cle(n):
    m = re.search(r"layers\.(\d+)\..*?(\w+_proj)\.(lora_[AB])", n)
    return m.groups() if m else None
par_cle = {cle(n): n for n in lora}

charges = 0
for k, v in poids.items():
    c = cle(k)
    if c in par_cle and lora[par_cle[c]].shape == v.shape:
        lora[par_cle[c]].data.copy_(v.to(device))
        charges += 1
print(f"{charges} / {len(lora)} matrices LoRA chargées")
print("Somme des lora_B :", sum(float(p.abs().sum()) for n, p in lora.items() if "lora_B" in n))

Clés du fichier : ['base_model.model.base_model.model.base_model.model.model.layers.0.mlp.down_proj.lora_A.weight', 'base_model.model.base_model.model.base_model.model.model.layers.0.mlp.down_proj.lora_B.weight']
336 / 336 matrices LoRA chargées
Somme des lora_B : 1226.6138784587383


/tmp/ipykernel_698/510598137.py:20: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print("Somme des lora_B :", sum(float(p.abs().sum()) for n, p in lora.items() if "lora_B" in n))


## 1. Les mêmes outils qu'en phase 8

Calculatrice, récompense, boucle agent : **exactement** le même code qu'à l'entraînement, pour que la comparaison soit juste. Une seule nouveauté : `avec_adaptateur`. Avec `False`, on **désactive** temporairement les matrices LoRA, et on retrouve le modèle d'avant GRPO, sans le recharger.

In [8]:
def calculer(expression):
    expression = expression.replace("×", "*").replace("x", "*").replace(" ", "")
    if not re.fullmatch(r"[\d+\-*/().]{1,60}", expression):
        return "erreur"
    try:
        r = eval(expression, {"__builtins__": {}})
        return str(int(r)) if float(r).is_integer() else f"{r:.4f}"
    except Exception:
        return "erreur"

def extraire(texte):
    m = re.findall(r"<reponse>\s*(-?[\d\s.,]+?)\s*</reponse>", texte)
    if not m:
        return None
    try:
        return int(m[-1].replace(" ", "").replace(".", "").replace(",", ""))
    except ValueError:
        return None

SYSTEME = """Tu résous des problèmes de calcul. Tu disposes d'une calculatrice.
Pour l'utiliser, écris une opération entre <calc> et </calc>, par exemple <calc>125*48</calc>.
Le résultat te sera donné entre <resultat> et </resultat>.
Quand tu as trouvé, donne la réponse finale (un nombre entier) entre <reponse> et </reponse>.

Exemple :
Problème : Un libraire vend 215 livres à 36 euros chacun. Combien gagne-t-il ?
<calc>215*36</calc><resultat>7740</resultat>
Il gagne 7740 euros. <reponse>7740</reponse>"""

@torch.no_grad()
def resoudre(problemes, avec_adaptateur=True, max_tours=4, max_tokens=160):
    trajs = []
    for enonce, attendu in problemes:
        msgs = [{"role": "system", "content": SYSTEME}, {"role": "user", "content": enonce}]
        prompt = tok(tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False),
                     add_special_tokens=False).input_ids
        trajs.append({"prompt": prompt, "suite": [], "texte": "", "attendu": attendu, "fini": False, "appels": 0})
    for tour in range(max_tours):
        actifs = [t for t in trajs if not t["fini"]]
        for k in range(0, len(actifs), 48):
            paquet = actifs[k:k + 48]
            entrees = [t["prompt"] + t["suite"] for t in paquet]
            L = max(map(len, entrees))
            x = torch.tensor([[tok.pad_token_id] * (L - len(e)) + e for e in entrees], device=device)
            masque = torch.zeros_like(x)
            for j, e in enumerate(entrees):
                masque[j, L - len(e):] = 1
            def generer():
                with torch.autocast("cuda", dtype=torch.float16):
                    return modele.generate(input_ids=x, attention_mask=masque, max_new_tokens=max_tokens,
                                           stop_strings=["</calc>"], tokenizer=tok,
                                           pad_token_id=tok.pad_token_id, do_sample=False)
            if avec_adaptateur:
                sortie = generer()
            else:
                with modele.disable_adapter():           # le modèle d'avant GRPO
                    sortie = generer()
            for t, s in zip(paquet, sortie[:, L:].tolist()):
                gen = []
                for i in s:
                    if i == tok.pad_token_id:
                        break
                    gen.append(i)
                texte = tok.decode(gen)
                t["suite"] += gen
                t["texte"] += texte
                appels = re.findall(r"<calc>(.*?)</calc>", texte, re.S)
                if appels and "<reponse>" not in texte and tour < max_tours - 1:
                    t["appels"] += 1
                    injecte = f"<resultat>{calculer(appels[-1])}</resultat>"
                    t["suite"] += tok(injecte, add_special_tokens=False).input_ids
                    t["texte"] += injecte
                else:
                    t["fini"] = True
    return trajs

def bilan(trajs):
    return {"exactitude": np.mean([extraire(t["texte"]) == t["attendu"] for t in trajs]),
            "format": np.mean([extraire(t["texte"]) is not None for t in trajs]),
            "outil": np.mean(["<calc>" in t["texte"] for t in trajs]),
            "appels moyens": np.mean([t["appels"] for t in trajs])}

## 2. Les quatre épreuves

**B** utilise des situations et des tournures **jamais vues** (prix, distances, abonnements…), mais les mêmes opérations.

**C** demande deux calculs enchaînés. Le modèle peut faire deux appels, ou tout mettre dans une seule expression comme `(a*b)-(c*d)` : les deux sont justes.

**D** pose des calculs qu'on fait de tête (« 7 + 5 »). Un bon usage d'outil, c'est aussi savoir **ne pas** s'en servir. Mais attention : on ne pénalise pas l'usage de l'outil, donc si le modèle l'utilise partout, ce n'est pas une « faute » au sens de la récompense. On observe seulement son comportement.

In [9]:
def epreuve_A(r):
    a, b, c = r.randint(120, 999), r.randint(13, 98), r.randint(10, 99)
    return r.choice([
        (f"Un fermier récolte {a} caisses contenant chacune {b} pommes. Combien de pommes a-t-il en tout ?", a * b),
        (f"Une salle compte {b} rangées de {a} sièges. {c} sièges sont occupés. Combien de sièges sont libres ?", a * b - c),
        (f"Un camion transporte {a} kg par voyage. Il fait {b} voyages, puis livre encore {c} kg. Combien de kg a-t-il transportés au total ?", a * b + c),
        (f"{b} amis se partagent équitablement {a * b} euros. Combien chacun reçoit-il ?", a),
        (f"Une usine fabrique {a} pièces par jour pendant {b} jours. Combien de pièces a-t-elle fabriquées ?", a * b),
    ])

def epreuve_B(r):
    a, b, c = r.randint(120, 999), r.randint(13, 98), r.randint(10, 99)
    return r.choice([
        (f"Quel est le prix total de {b} billets de train à {a} euros l'unité ?", a * b),
        (f"Une coureuse parcourt {a} mètres à chaque tour de piste. Elle fait {b} tours. Quelle distance a-t-elle courue, en mètres ?", a * b),
        (f"Un abonnement coûte {a} euros par mois. Combien paie-t-on en {b} mois, sachant qu'on reçoit une réduction de {c} euros ?", a * b - c),
        (f"On répartit {a * b} litres d'eau dans {b} citernes identiques. Combien de litres contient chaque citerne ?", a),
        (f"Calcule le produit de {a} par {b}.", a * b),
        (f"Si un livre a {a} pages et qu'une bibliothèque possède {b} exemplaires de ce livre, combien de pages cela fait-il en tout ?", a * b),
    ])

def epreuve_C(r):
    a, b, c, d, k = r.randint(120, 999), r.randint(13, 98), r.randint(120, 999), r.randint(13, 98), r.randint(3, 9)
    return r.choice([
        (f"Un magasin vend {b} chaises à {a} euros et {d} tables à {c} euros. Quel est le montant total ?", a * b + c * d),
        (f"Une école commande {b} cartons de {a} cahiers. Elle donne {d} cahiers à chacune de ses {k} classes. Combien de cahiers reste-t-il ?", a * b - d * k),
        (f"Un car transporte {b} passagers par trajet et fait {d} trajets par jour. Combien de passagers transporte-t-il en {k} jours ?", b * d * k),
    ])

def epreuve_D(r):
    a, b = r.randint(2, 9), r.randint(2, 9)
    return r.choice([(f"Combien font {a} plus {b} ?", a + b), (f"Combien font {a} fois {b} ?", a * b),
                     (f"J'ai {a} pommes et j'en mange une. Combien m'en reste-t-il ?", a - 1)])

r = random.Random(2026)
epreuves = {
    "A. mêmes formulations": [epreuve_A(r) for _ in range(60)],
    "B. formulations nouvelles": [epreuve_B(r) for _ in range(60)],
    "C. deux étapes": [epreuve_C(r) for _ in range(60)],
    "D. calculs faciles": [epreuve_D(r) for _ in range(60)],
}
for nom, pbs in epreuves.items():
    print(f"{nom:<28} ex. : {pbs[0][0]}  -> {pbs[0][1]}")

A. mêmes formulations        ex. : Une usine fabrique 241 pièces par jour pendant 53 jours. Combien de pièces a-t-elle fabriquées ?  -> 12773
B. formulations nouvelles    ex. : Une coureuse parcourt 680 mètres à chaque tour de piste. Elle fait 64 tours. Quelle distance a-t-elle courue, en mètres ?  -> 43520
C. deux étapes               ex. : Un magasin vend 14 chaises à 980 euros et 44 tables à 762 euros. Quel est le montant total ?  -> 47248
D. calculs faciles           ex. : Combien font 7 fois 7 ?  -> 49


## 3. Le verdict

In [10]:
resultats, trajectoires = [], {}
t0 = time.time()
for nom, pbs in epreuves.items():
    for avec, etiquette in [(False, "avant GRPO"), (True, "après GRPO")]:
        trajs = resoudre(pbs, avec_adaptateur=avec)
        trajectoires[(nom, etiquette)] = trajs
        resultats.append({"épreuve": nom, "modèle": etiquette, **bilan(trajs)})
    print(f"{nom} : fait ({time.time() - t0:.0f} s)")

import pandas as pd
df = pd.DataFrame(resultats)
print()
print(df.to_string(index=False, formatters={"exactitude": "{:.0%}".format, "format": "{:.0%}".format,
                                            "outil": "{:.0%}".format, "appels moyens": "{:.1f}".format}))

A. mêmes formulations : fait (39 s)
B. formulations nouvelles : fait (78 s)
C. deux étapes : fait (113 s)
D. calculs faciles : fait (134 s)

                  épreuve     modèle exactitude format outil appels moyens
    A. mêmes formulations avant GRPO         2%    13%   70%           0.8
    A. mêmes formulations après GRPO       100%   100%  100%           1.0
B. formulations nouvelles avant GRPO         8%    17%   37%           0.4
B. formulations nouvelles après GRPO        92%   100%  100%           1.0
           C. deux étapes avant GRPO         0%     7%   33%           0.3
           C. deux étapes après GRPO        68%   100%  100%           1.0
       D. calculs faciles avant GRPO         0%     0%   80%           0.8
       D. calculs faciles après GRPO       100%   100%  100%           1.0


## 4. Regarder les erreurs

Pour chaque épreuve, quelques échecs du modèle **après GRPO**, avec la catégorie d'erreur. C'est là qu'on comprend *ce qu'il a appris* et *ce qu'il n'a pas appris*.

In [11]:
def categorie(t):
    resultats = [int(x) for x in re.findall(r"<resultat>(-?\d+)</resultat>", t["texte"])]
    if extraire(t["texte"]) is None:
        return "format"
    if "<calc>" not in t["texte"]:
        return "pas d'outil"
    if t["attendu"] in resultats:
        return "mauvaise recopie"
    return "mauvaise opération"

for nom in epreuves:
    trajs = trajectoires[(nom, "après GRPO")]
    erreurs = [t for t in trajs if extraire(t["texte"]) != t["attendu"]]
    compte = pd.Series([categorie(t) for t in erreurs]).value_counts().to_dict() if erreurs else {}
    print(f"\n=== {nom} : {len(erreurs)} erreurs {compte}")
    for t in erreurs[:2]:
        enonce = epreuves[nom][trajs.index(t)][0]
        print(f"  ❓ {enonce} (attendu {t['attendu']})")
        print(f"     {t['texte'][:220].replace(chr(10), ' ')}")


=== A. mêmes formulations : 0 erreurs {}

=== B. formulations nouvelles : 5 erreurs {'mauvaise opération': 5}
  ❓ Un abonnement coûte 797 euros par mois. Combien paie-t-on en 36 mois, sachant qu'on reçoit une réduction de 59 euros ? (attendu 28633)
     <calc>(797*36)-797</calc><<resultat>27895</resultat> <reponse>27895</reponse>.<|im_end|>
  ❓ Un abonnement coûte 228 euros par mois. Combien paie-t-on en 77 mois, sachant qu'on reçoit une réduction de 50 euros ? (attendu 17506)
     <calc>(228*77)/100</calc><<resultat>175.5600</resultat> <reponse>175.5600</reponse>.<|im_end|>

=== C. deux étapes : 19 erreurs {'mauvaise opération': 19}
  ❓ Une école commande 78 cartons de 347 cahiers. Elle donne 64 cahiers à chacune de ses 8 classes. Combien de cahiers reste-t-il ? (attendu 26554)
     <calc>(78*347)/64</calc><<resultat>422.9062</resultat> <reponse>422.9062</reponse>.<|im_end|>
  ❓ Une école commande 87 cartons de 342 cahiers. Elle donne 86 cahiers à chacune de ses 3 classes. Combien de

## 5. Et hors des mathématiques ?

Une dernière observation, sans note : que fait le modèle GRPO face à une question qui n'a **rien** d'un calcul ? S'il dégaine la calculatrice pour « Quelle est la capitale de la France ? », c'est que le renforcement a créé un **réflexe** plutôt qu'une compétence.

In [12]:
hors_sujet = [("Quelle est la capitale de la France ?", None), ("Écris une phrase sur la mer.", None),
              ("Quel est le contraire de « chaud » ?", None)]
for avec, etiquette in [(False, "AVANT"), (True, "APRÈS")]:
    print(f"=== {etiquette} GRPO ===")
    for t, (q, _) in zip(resoudre(hors_sujet, avec_adaptateur=avec, max_tokens=60), hors_sujet):
        print(f"  {q} -> {t['texte'][:150].replace(chr(10), ' ')}")

=== AVANT GRPO ===
  Quelle est la capitale de la France ? -> <calc>Paris</calc><<resultat>erreur</resultat>></calc>
  Écris une phrase sur la mer. -> "La mer est un immense fleuve qui coule dans le monde."<|im_end|>
  Quel est le contraire de « chaud » ? -> « Froid ».<|im_end|>
=== APRÈS GRPO ===
  Quelle est la capitale de la France ? -> <calc>Paris</calc><<resultat>erreur</resultat>  <reponse>erreur</reponse>.<|im_end|>
  Écris une phrase sur la mer. -> <calc>mer</calc><<resultat>erreur</resultat> <reponse>erreur</reponse>.<|im_end|>
  Quel est le contraire de « chaud » ? -> <calc>-chau</calc><<resultat>erreur</resultat> <reponse>erreur</reponse>.<|im_end|>
